In [1]:
from google.colab import drive
import os

# Mount your Google Drive
drive.mount('/content/drive')

# Verify your uploaded folder exists
DATA_ROOT = '/content/drive/MyDrive/fundus dataset'
print("Folders inside your dataset directory:", os.listdir(DATA_ROOT))

ModuleNotFoundError: No module named 'google'

In [ ]:
import os

# Update the path to include the nested subfolder
DATA_ROOT = '/content/drive/MyDrive/fundus dataset/fundus dataset'

print("Folders inside your dataset directory:")
print(os.listdir(DATA_ROOT))

Folders inside your dataset directory:
['test', 'train', 'Quality Assessment.xlsx']


In [ ]:
import os

# Base root path
DATA_ROOT = '/content/drive/MyDrive/fundus dataset/fundus dataset'

# Paths to train and test
train_path = os.path.join(DATA_ROOT, 'train')
test_path = os.path.join(DATA_ROOT, 'test')

print("Inside 'train':", os.listdir(train_path))
print("Inside 'test':", os.listdir(test_path))

Inside 'train': ['Ground truth', 'Original']
Inside 'test': ['Original', 'Ground truth']


In [ ]:
%%writefile config.py
import torch

class Config:
    # Base dataset path in Google Drive
    DATA_ROOT = '/content/drive/MyDrive/fundus dataset/fundus dataset'

    # Pointing to the folders with your actual images and masks
    # (If your main data is currently under 'test', we can use it for a quick full test run)
    TRAIN_IMG_DIR = f"{DATA_ROOT}/test/Original"
    TRAIN_MASK_DIR = f"{DATA_ROOT}/test/Ground truth"

    VAL_IMG_DIR = f"{DATA_ROOT}/test/Original"
    VAL_MASK_DIR = f"{DATA_ROOT}/test/Ground truth"

    # Model & Data Hyperparameters
    IMAGE_HEIGHT = 512
    IMAGE_WIDTH = 512
    BATCH_SIZE = 4
    EPOCHS = 10
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 2

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Overwriting config.py


In [ ]:
%%writefile dataset.py
import os
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

class FIVESDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(512, 512)):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif')

        # Get all mask filenames into a dictionary mapped by base filename
        all_masks = {os.path.splitext(f)[0]: f for f in os.listdir(mask_dir) if f.lower().endswith(valid_exts)}
        all_images = [f for f in os.listdir(image_dir) if f.lower().endswith(valid_exts)]

        self.images = []
        self.masks = []

        # Match images directly to masks with matching names
        for img_file in sorted(all_images):
            base_name = os.path.splitext(img_file)[0]
            if base_name in all_masks:
                self.images.append(img_file)
                self.masks.append(all_masks[base_name])

        print(f"Dataset Loaded: Matched exactly {len(self.images)} images and {len(self.masks)} masks!")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = image.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32) / 255.0

        image = np.transpose(image, (2, 0, 1))
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

Overwriting dataset.py


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        # Encoder (Downsampling)
        self.enc1 = ConvBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.enc2 = ConvBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = ConvBlock(128, 256)

        # Decoder (Upsampling)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        # Final output layer
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        d2 = self.up2(b)
        d2 = torch.cat((e2, d2), dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat((e1, d1), dim=1)
        d1 = self.dec1(d1)

        return self.final_conv(d1)

Writing model.py


In [ ]:
%%writefile train.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from config import Config
from dataset import FIVESDataset
from model import UNet

def train_fives_model():
    print(f"Starting training on device: {Config.DEVICE}")

    # 1. Datasets & Loaders
    train_dataset = FIVESDataset(
        Config.TRAIN_IMG_DIR,
        Config.TRAIN_MASK_DIR,
        img_size=(Config.IMAGE_HEIGHT, Config.IMAGE_WIDTH)
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )

    # 2. Initialize Model, Loss Function, and Optimizer
    model = UNet(in_channels=3, out_channels=1).to(Config.DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)

    # 3. Training Loop
    os.makedirs("weights", exist_ok=True)
    best_loss = float("inf")

    for epoch in range(Config.EPOCHS):
        model.train()
        running_loss = 0.0

        for images, masks in train_loader:
            images, masks = images.to(Config.DEVICE), masks.to(Config.DEVICE)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, masks)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch [{epoch+1}/{Config.EPOCHS}] | Training Loss: {epoch_loss:.4f}")

        # Save best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), "weights/unet_fives_best.pth")

    print("Training finished! Best model saved to weights/unet_fives_best.pth")

if __name__ == "__main__":
    train_fives_model()

Writing train.py


In [4]:
!python train.py

python: can't open file 'c:\\Users\\HIMANSHI SHAH\\Downloads\\fundus dataset\\train.py': [Errno 2] No such file or directory


In [3]:
%%writefile evaluate.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from config import Config
from dataset import FIVESDataset
from model import UNet

def calculate_metrics(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()

    intersection = (pred * target).sum()
    total = pred.sum() + target.sum()

    dice = (2. * intersection + 1e-8) / (total + 1e-8)
    iou = (intersection + 1e-8) / (pred.sum() + target.sum() - intersection + 1e-8)

    return dice.item(), iou.item()

def evaluate():
    val_dataset = FIVESDataset(
        Config.VAL_IMG_DIR,
        Config.VAL_MASK_DIR,
        img_size=(Config.IMAGE_HEIGHT, Config.IMAGE_WIDTH)
    )
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

    model = UNet(in_channels=3, out_channels=1).to(Config.DEVICE)
    model.load_state_dict(torch.load("weights/unet_fives_best.pth", map_location=Config.DEVICE))
    model.eval()

    total_dice, total_iou = 0.0, 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(Config.DEVICE), masks.to(Config.DEVICE)
            outputs = model(images)

            dice, iou = calculate_metrics(outputs, masks)
            total_dice += dice * images.size(0)
            total_iou += iou * images.size(0)

    avg_dice = total_dice / len(val_dataset)
    avg_iou = total_iou / len(val_dataset)

    print("=" * 40)
    print(f"EVALUATION RESULTS:")
    print(f"Mean Dice Score (F1): {avg_dice:.4f}")
    print(f"Mean IoU (Jaccard):    {avg_iou:.4f}")
    print("=" * 40)

if __name__ == "__main__":
    evaluate()

Writing evaluate.py


In [2]:
!python evaluate.py

python: can't open file 'c:\\Users\\HIMANSHI SHAH\\Downloads\\fundus dataset\\evaluate.py': [Errno 2] No such file or directory


In [ ]:
import os

path = '/content/drive/MyDrive/fundus dataset/fundus dataset/train/Original'
all_files = os.listdir(path)

print(f"Total raw items in folder: {len(all_files)}")
# Print distinct file extensions found
extensions = set([os.path.splitext(f)[1].lower() for f in all_files])
print("Extensions found:", extensions)
print("First 10 items:", all_files[:10])

Total raw items in folder: 12
Extensions found: {'.db', '.png'}
First 10 items: ['373_G.png', '384_G.png', '361_G.png', '360_G.png', '447_G.png', '326_G.png', '364_G.png', '353_G.png', '341_G.png', '448_G.png']


In [ ]:
import os

path = '/content/drive/MyDrive/fundus dataset/fundus dataset/train/Original'
if os.path.exists(path):
    print("Files found:", os.listdir(path)[:5]) # Prints first 5 files
else:
    print("Path does not exist!")

Files found: ['373_G.png', '384_G.png', '361_G.png', '360_G.png', '447_G.png']


In [ ]:
import os

# Search your entire Google Drive dataset folder for any folder named 'Original'
base_search_path = '/content/drive/MyDrive/fundus dataset'

for root, dirs, files in os.walk(base_search_path):
    if 'Original' in dirs:
        full_path = os.path.join(root, 'Original')
        print(f"Found 'Original' folder at: {full_path} | Contains {len(os.listdir(full_path))} files")

Found 'Original' folder at: /content/drive/MyDrive/fundus dataset/fundus dataset/test/Original | Contains 134 files
Found 'Original' folder at: /content/drive/MyDrive/fundus dataset/fundus dataset/train/Original | Contains 12 files
